# The Blockbuster Formula
### Bayesian Network Analysis of Box Office Success (2000–2025)

## Section 0 - Setup & Imports

In [12]:
import os
import re
import time
import warnings
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx
from pathlib import Path
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from bs4 import BeautifulSoup

# pgmpy
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator, PC, BIC
from pgmpy.inference import VariableElimination
import pgmpy

# sklearn
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score,
    multilabel_confusion_matrix
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# Paths
DATA_PATH    = Path('data')
OUTPUTS_PATH = Path('outputs')
DATA_PATH.mkdir(exist_ok=True)
OUTPUTS_PATH.mkdir(exist_ok=True)

# Colour palette used across all plots
OUTCOME_COLORS = {
    'Flop':        '#e74c3c',
    'Break-even':  '#f39c12',
    'Hit':         '#2ecc71',
    'Blockbuster': '#2980b9',
}
outcome_order = ['Flop', 'Break-even', 'Hit', 'Blockbuster']

# API
load_dotenv()
TMDB_API_KEY = os.getenv('TMDB_API_KEY')
if not TMDB_API_KEY:
    raise ValueError('TMDb API key not found. Add TMDB_API_KEY to your .env file.')

print(f'pgmpy {pgmpy.__version__} | pandas {pd.__version__} | numpy {np.__version__}')
print('All imports OK.')

pgmpy 1.1.0 | pandas 2.2.3 | numpy 2.2.3
All imports OK.


## Section 1 - Data Collection

In [13]:
# TMDb API config and helper
TMDB_BASE    = 'https://api.themoviedb.org/3'
TMDB_HEADERS = {'accept': 'application/json'}
START_YEAR   = 2000
END_YEAR     = 2025
PAGES_PER_YEAR = 10
REQUEST_DELAY  = 0.25

RAW_PATH  = DATA_PATH / 'movies_raw_v2.csv'
SUPP_PATH = DATA_PATH / 'numbers_supplement_v2.csv'
FEAT_PATH = DATA_PATH / 'movies_featured_v2.csv'

def tmdb_get(endpoint, params=None):
    url = f'{TMDB_BASE}/{endpoint}'
    all_params = {'api_key': TMDB_API_KEY}
    if params:
        all_params.update(params)
    try:
        r = requests.get(url, headers=TMDB_HEADERS, params=all_params, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.RequestException as e:
        print(f'Request failed for {endpoint}: {e}')
        return None

test = tmdb_get('configuration')
print('TMDb connection:', 'OK' if test else 'FAILED -- check API key')


TMDb connection: OK


In [9]:
# Smart loader -- skip API calls if raw data already exists
if RAW_PATH.exists():
    df_raw = pd.read_csv(RAW_PATH, parse_dates=['release_date'])
    print(f'Raw data already exists -- loaded {len(df_raw)} rows from {RAW_PATH}')
    print(f'Date range: {df_raw["release_date"].min().year} - {df_raw["release_date"].max().year}')
    print('Skip to the Numbers supplement cell below.')
else:
    print(f'{RAW_PATH} not found -- run the cells below to fetch from TMDb.')
    df_raw = None


data\movies_raw_v2.csv not found -- run the cells below to fetch from TMDb.


In [14]:
# Step 1: Discover movie IDs per year -- skip if raw data already loaded
if df_raw is not None:
    print('Raw data already loaded -- skipping.')
else:
    def discover_ids(year, pages=PAGES_PER_YEAR):
        ids = []
        for page in range(1, pages + 1):
            data = tmdb_get('discover/movie', params={
                'primary_release_year': year, 'sort_by': 'popularity.desc',
                'include_adult': False, 'include_video': False,
                'page': page, 'language': 'en-US', 'with_original_language': 'en',
            })
            if data and 'results' in data:
                ids.extend([m['id'] for m in data['results']])
            time.sleep(REQUEST_DELAY)
        return ids

    print(f'Discovering IDs {START_YEAR} - {END_YEAR}...')
    all_ids = []
    for year in tqdm(range(START_YEAR, END_YEAR + 1), desc='Years'):
        all_ids.extend(discover_ids(year))
    all_ids = list(set(all_ids))
    print(f'Unique IDs collected: {len(all_ids)}')


Discovering IDs 2000 - 2025...


Years:   0%|          | 0/26 [00:00<?, ?it/s]

Unique IDs collected: 4741


In [15]:
# Step 2: Fetch details + credits -- skip if raw data already loaded
if df_raw is not None:
    print('Raw data already loaded -- skipping.')
else:
    def fetch_details(movie_id):
        details = tmdb_get(f'movie/{movie_id}', params={'language': 'en-US'})
        if not details:
            return None
        credits = tmdb_get(f'movie/{movie_id}/credits', params={'language': 'en-US'})
        cast_list, cast_ids = [], []
        if credits and 'cast' in credits:
            top = sorted(credits['cast'], key=lambda x: x.get('order', 99))[:5]
            cast_list = [c['name'] for c in top]
            cast_ids  = [str(c['id']) for c in top]
        return {
            'tmdb_id': movie_id,
            'title': details.get('title', ''),
            'release_date': details.get('release_date', ''),
            'genres': '|'.join(g['name'] for g in details.get('genres', [])),
            'budget': details.get('budget', 0),
            'revenue': details.get('revenue', 0),
            'runtime': details.get('runtime', 0),
            'popularity': details.get('popularity', 0),
            'vote_average': details.get('vote_average', 0),
            'vote_count': details.get('vote_count', 0),
            'original_language': details.get('original_language', ''),
            'cast_names': '|'.join(cast_list),
            'cast_ids':   '|'.join(cast_ids),
        }

    print(f'Fetching details for {len(all_ids)} movies...')
    movies, failed = [], []
    for mid in tqdm(all_ids, desc='Movies'):
        rec = fetch_details(mid)
        if rec: movies.append(rec)
        else:   failed.append(mid)
        time.sleep(REQUEST_DELAY)   
    print(f'Fetched: {len(movies)}  |  Failed: {len(failed)}')


Fetching details for 4741 movies...


Movies:   0%|          | 0/4741 [00:00<?, ?it/s]

Fetched: 4741  |  Failed: 0


In [16]:
# Step 3: Build DataFrame and save
if df_raw is not None:
    print('Raw data already loaded -- skipping build.')
else:
    df_raw = pd.DataFrame(movies)
    df_raw['release_date'] = pd.to_datetime(df_raw['release_date'], errors='coerce')
    df_raw['release_year'] = df_raw['release_date'].dt.year
    df_raw.to_csv(RAW_PATH, index=False)
    print(f'Saved {len(df_raw)} rows to {RAW_PATH}')

print(f'Shape: {df_raw.shape}')
df_raw.head(3)


Saved 4741 rows to data\movies_raw_v2.csv
Shape: (4741, 14)


,tmdb_id,title,release_date,genres,budget,revenue,runtime,popularity,vote_average,vote_count,original_language,cast_names,cast_ids,release_year
0,8193,Napoleon Dynamite,2004-06-11,Comedy,400000,46118097,95,3.4054,6.783,2127,en,Jon Heder|Efren Ramirez|Tina Majorino|Aaron Ru...,53926|20190|53930|53927|9629,2004
1,8198,The Quiet American,2002-11-22,Romance|Thriller|Drama|War,30000000,27674124,101,1.5456,6.545,325,en,Michael Caine|Brendan Fraser|Đỗ Thị Hải Yến|Tz...,3895|18269|53962|21629|1118,2002
2,614409,To All the Boys: Always and Forever,2021-02-12,Romance|Comedy|Drama,0,0,114,2.8253,7.502,2054,en,Lana Condor|Noah Centineo|Janel Parrish|Anna C...,1452046|1253353|93377|1683266|1299232,2021


In [17]:
# Raw data overview
print('=== Raw Data Summary ===')
print(f'Total rows:         {len(df_raw)}')
print(f'Budget  > 0:        {(df_raw["budget"] > 0).sum()}')
print(f'Revenue > 0:        {(df_raw["revenue"] > 0).sum()}')
print(f'Both zero:          {((df_raw["budget"]==0) & (df_raw["revenue"]==0)).sum()}')
print(f'Date range:         {df_raw["release_date"].min().date()} to {df_raw["release_date"].max().date()}')
bnz = df_raw[df_raw['budget']  > 0]['budget']
rnz = df_raw[df_raw['revenue'] > 0]['revenue']
print(f'Budget  min/med/max: ${bnz.min():,.0f} / ${bnz.median():,.0f} / ${bnz.max():,.0f}')
print(f'Revenue min/med/max: ${rnz.min():,.0f} / ${rnz.median():,.0f} / ${rnz.max():,.0f}')


=== Raw Data Summary ===
Total rows:         4741
Budget  > 0:        3551
Revenue > 0:        3599
Both zero:          869
Date range:         2000-01-01 to 2025-12-28
Budget  min/med/max: $5 / $30,000,000 / $489,900,000
Revenue min/med/max: $7 / $54,700,000 / $2,923,706,026


In [ ]:
# The Numbers budget supplement
# Fills in missing production budgets for films TMDb has at $0 but revenue > $5M.
# Cached to SUPP_PATH. Set FORCE_REFETCH=True to re-scrape.

FORCE_REFETCH = False
TN_BASE    = 'https://www.the-numbers.com/movie/'
TN_HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

def _tn_slug(title, year=None):
    t = title.strip()
    for art in ['The ', 'A ', 'An ']:
        if t.startswith(art):
            t = t[len(art):] + '-' + art.strip()
            break
    t = re.sub(r"[\u2019\u2018']", '', t)
    t = re.sub(r'[^a-zA-Z0-9-]', '-', t)
    t = re.sub(r'-+', '-', t).strip('-')
    return f'{t}-({year})' if year else t

def _tn_parse(html):
    text = html.replace('\xa0', ' ')
    soup = BeautifulSoup(text, 'html.parser')
    budget = revenue = None
    m = re.search(r'Production Budget:\s*\$([\d,]+)', text)
    if m:
        budget = int(m.group(1).replace(',', ''))
    for row in soup.find_all('tr'):
        if 'Worldwide Box Office' in row.get_text():
            m2 = re.search(r'\$([\d,]+)', row.get_text())
            if m2:
                revenue = int(m2.group(1).replace(',', ''))
                break
    return budget, revenue

def _tn_fetch(title, year):
    slugs = list(dict.fromkeys([
        _tn_slug(title), _tn_slug(title, year),
        _tn_slug(title.replace('&', 'and')), _tn_slug(title.replace('&', 'and'), year),
    ]))
    for slug in slugs:
        try:
            r = requests.get(TN_BASE + slug, headers=TN_HEADERS, timeout=8)
            if r.status_code == 200:
                b, rev = _tn_parse(r.text)
                if b or rev:
                    return b, rev
        except Exception:
            pass
        time.sleep(0.25)
    return None, None

if SUPP_PATH.exists() and not FORCE_REFETCH:
    df_supp = pd.read_csv(SUPP_PATH)
    print(f'Cached supplement loaded: {SUPP_PATH}')
else:
    _tmp = df_raw.copy()
    _tmp['budget']  = pd.to_numeric(_tmp['budget'],  errors='coerce').fillna(0)
    _tmp['revenue'] = pd.to_numeric(_tmp['revenue'], errors='coerce').fillna(0)
    _tmp['release_year'] = pd.to_datetime(_tmp['release_date'], errors='coerce').dt.year
    targets = _tmp[(_tmp['budget'] == 0) & (_tmp['revenue'] > 5_000_000)]
    print(f'Fetching {len(targets)} films from The Numbers...')
    rows = []
    for _, row in tqdm(targets.iterrows(), total=len(targets)):
        yr = int(row['release_year']) if pd.notna(row['release_year']) else None
        b, rev = _tn_fetch(row['title'], yr)
        rows.append({'tmdb_id': row['tmdb_id'], 'title': row['title'],
                     'numbers_budget': b, 'numbers_revenue': rev})
        time.sleep(0.4)
    df_supp = pd.DataFrame(rows)
    df_supp.to_csv(SUPP_PATH, index=False)
    print(f'Saved to {SUPP_PATH}')

matched = df_supp[df_supp['numbers_budget'].notna()]
print(f'Budgets recovered: {len(matched)} / {len(df_supp)}')
top = matched[['title','numbers_budget']].sort_values('numbers_budget', ascending=False).head(10)
print(top.to_string(index=False))

## Section 2 - Data Cleaning

In [20]:
# Load raw data and merge The Numbers supplement to fill missing budgets
if 'df_raw' not in dir() or df_raw is None:
    df_raw = pd.read_csv(RAW_PATH, parse_dates=['release_date'])
    print(f'Loaded {len(df_raw)} rows from {RAW_PATH}')

df = df_raw.copy()
df['budget']      = pd.to_numeric(df['budget'],  errors='coerce').fillna(0)
df['revenue']     = pd.to_numeric(df['revenue'], errors='coerce').fillna(0)
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year

# Merge supplement
if 'df_supp' not in dir() or df_supp is None:
    df_supp = pd.read_csv(SUPP_PATH)
    print(f'Loaded supplement: {len(df_supp)} rows')

df = df.merge(df_supp[['tmdb_id', 'numbers_budget', 'numbers_revenue']], on='tmdb_id', how='left')

# Patch zeros with The Numbers values where available
mask_b = (df['budget']  == 0) & df['numbers_budget'].notna()
mask_r = (df['revenue'] == 0) & df['numbers_revenue'].notna()
df.loc[mask_b, 'budget']  = df.loc[mask_b, 'numbers_budget']
df.loc[mask_r, 'revenue'] = df.loc[mask_r, 'numbers_revenue']
df.drop(columns=['numbers_budget', 'numbers_revenue'], inplace=True)

print(f'Budget  > 0: {(df["budget"]  > 0).sum()}  (was {(df_raw["budget"]  > 0).sum()})')
print(f'Revenue > 0: {(df["revenue"] > 0).sum()}  (was {(df_raw["revenue"] > 0).sum()})')

Budget  > 0: 3551  (was 3551)
Revenue > 0: 3599  (was 3599)


In [21]:
# Inflate all financials to 2024 dollars using US CPI-U annual averages
CPI_BASE = 314.796  # 2024 annual average

CPI_TABLE = {
    2000: 172.2,   2001: 177.1,   2002: 179.9,   2003: 184.0,   2004: 188.9,
    2005: 195.3,   2006: 201.6,   2007: 207.3,   2008: 215.3,   2009: 214.5,
    2010: 218.1,   2011: 224.9,   2012: 229.6,   2013: 233.0,   2014: 236.7,
    2015: 237.0,   2016: 240.0,   2017: 245.1,   2018: 251.1,   2019: 255.7,
    2020: 258.8,   2021: 270.970, 2022: 292.655, 2023: 304.702,
    2024: 314.796, 2025: 319.1,
}

df['cpi']         = df['release_year'].map(CPI_TABLE).fillna(CPI_BASE)
df['budget_adj']  = (df['budget']  * CPI_BASE / df['cpi']).round(0)
df['revenue_adj'] = (df['revenue'] * CPI_BASE / df['cpi']).round(0)

mult_2000 = CPI_BASE / CPI_TABLE[2000]
print(f'Inflation base: 2024 (CPI = {CPI_BASE})')
print(f'2000→2024 multiplier: {mult_2000:.3f}x  (e.g. $100M → ${100_000_000 * mult_2000:,.0f})')

Inflation base: 2024 (CPI = 314.796)
2000→2024 multiplier: 1.828x  (e.g. $100M → $182,808,362)


In [22]:
# Filter to films with known financials; remove data-entry errors only
# No minimum budget floor -- inflation adjustment handles era differences
n_before = len(df)

df_clean = df[
    (df['budget_adj'] > 0) &
    (df['revenue_adj'] > 0) &
    (df['budget_adj']  < 700_000_000) &   # no film has ever cost >$700M adj.
    (df['revenue_adj'] < 6_000_000_000)   # highest ever ~$2.9B adj.
    ].copy()

print(f'Rows before filter: {n_before}')
print(f'Rows after  filter: {len(df_clean)}  ({n_before - len(df_clean)} removed)')
print(f'Year range: {df_clean["release_year"].min():.0f} – {df_clean["release_year"].max():.0f}')

Rows before filter: 4741
Rows after  filter: 3278  (1463 removed)
Year range: 2000 – 2025


In [23]:
# Cleaned data overview
print('=== Cleaned Dataset ===')
print(f'Total films:  {len(df_clean)}')
bnz = df_clean['budget_adj']
rnz = df_clean['revenue_adj']
print(f'Budget  (adj) min/med/max: ${bnz.min():,.0f} / ${bnz.median():,.0f} / ${bnz.max():,.0f}')
print(f'Revenue (adj) min/med/max: ${rnz.min():,.0f} / ${rnz.median():,.0f} / ${rnz.max():,.0f}')
print(f'\nFilms per year:')
print(df_clean.groupby('release_year').size().rename('count').to_string())

=== Cleaned Dataset ===
Total films:  3278
Budget  (adj) min/med/max: $7 / $45,132,043 / $603,123,036
Revenue (adj) min/med/max: $7 / $93,596,301 / $4,290,773,716

Films per year:
release_year
2000    130
2001    144
2002    145
2003    143
2004    148
2005    152
2006    144
2007    151
2008    140
2009    136
2010    141
2011    142
2012    135
2013    140
2014    142
2015    123
2016    131
2017    123
2018    124
2019    118
2020     64
2021     78
2022     77
2023     93
2024    110
2025    104


## Section 3 - Feature Engineering

In [24]:
# Outcome label: Flop / Break-even / Hit / Blockbuster
# Thresholds: a film needs ~2.5x production budget to break even after
# theater splits (~50%) and P&A costs (~= production budget)
df_clean['profit_ratio'] = df_clean['revenue_adj'] / df_clean['budget_adj']

OUTCOME_BINS   = [0, 1.0, 2.5, 5.0, np.inf]
OUTCOME_LABELS_DEF = ['Flop', 'Break-even', 'Hit', 'Blockbuster']

df_clean['outcome_label'] = pd.cut(
    df_clean['profit_ratio'],
    bins=OUTCOME_BINS, labels=OUTCOME_LABELS_DEF, right=False
)

print('Outcome distribution:')
for label in outcome_order:
    count = (df_clean['outcome_label'] == label).sum()
    pct   = count / len(df_clean) * 100
    print(f'  {label:<12} {count:>4}  ({pct:.1f}%)')

Outcome distribution:
  Flop          863  (26.3%)
  Break-even    996  (30.4%)
  Hit           845  (25.8%)
  Blockbuster   574  (17.5%)


In [25]:
# Budget tiers (2024-adjusted dollars)
BUDGET_BINS   = [0, 10e6, 40e6, 100e6, 200e6, np.inf]
BUDGET_LABELS = ['Micro', 'Low', 'Mid', 'High', 'Mega']

df_clean['budget_tier'] = pd.cut(
    df_clean['budget_adj'],
    bins=BUDGET_BINS, labels=BUDGET_LABELS, right=False
)

print('Budget tier distribution:')
for tier, count in df_clean['budget_tier'].value_counts()[BUDGET_LABELS].items():
    pct = count / len(df_clean) * 100
    print(f'  {tier:<8} {count:>4}  ({pct:.1f}%)')

Budget tier distribution:
  Micro     381  (11.6%)
  Low      1110  (33.9%)
  Mid       998  (30.4%)
  High      533  (16.3%)
  Mega      256  (7.8%)


In [26]:
# Map TMDb genres → 5 BN-friendly categories using the first listed genre
GENRE_MAP = {
    'Action':           'Action',  'Adventure':  'Action',  'Thriller':  'Action',
    'War':              'Action',  'Western':    'Action',
    'Comedy':           'Comedy',  'Animation':  'Comedy',  'Family':    'Comedy',
    'Music':            'Comedy',
    'Drama':            'Drama',   'History':    'Drama',   'Romance':   'Drama',
    'Horror':           'Horror',  'Mystery':    'Horror',  'Crime':     'Horror',
    'Science Fiction':  'Sci-Fi',  'Fantasy':    'Sci-Fi',
}

def primary_genre(genres_str):
    if pd.isna(genres_str) or genres_str == '':
        return 'Other'
    first = genres_str.split('|')[0].strip()
    return GENRE_MAP.get(first, 'Other')

df_clean['genre_bn'] = df_clean['genres'].apply(primary_genre)

print('Genre BN distribution:')
print(df_clean['genre_bn'].value_counts().to_string())

Genre BN distribution:
genre_bn
Action    965
Comedy    845
Drama     772
Horror    485
Sci-Fi    203
Other       8


In [27]:
# Release window (season) from release month
def release_window(month):
    if pd.isna(month): return 'Other'
    m = int(month)
    if m in (6, 7, 8):   return 'Summer'
    if m in (11, 12):    return 'Holiday'
    if m in (3, 4, 5):   return 'Spring'
    return 'Other'   # Jan, Feb, Sep, Oct

df_clean['release_window'] = df_clean['release_date'].dt.month.apply(release_window)

print('Release window distribution:')
print(df_clean['release_window'].value_counts().to_string())

Release window distribution:
release_window
Other      1128
Summer      775
Spring      743
Holiday     632


In [28]:
import json

# Actor prestige: log10 of lead actor's mean career revenue (adj.)
df_clean['lead_actor'] = df_clean['cast_names'].apply(
    lambda x: x.split('|')[0].strip() if pd.notna(x) and x != '' else 'Unknown'
)

actor_stats = (
    df_clean.groupby('lead_actor')
    .agg(films=('title', 'count'), avg_rev=('revenue_adj', 'mean'))
    .query('films >= 2')
    .copy()
)
actor_stats['prestige_score'] = np.log10(actor_stats['avg_rev'].clip(lower=1))

q25, q50, q75 = actor_stats['prestige_score'].quantile([0.25, 0.50, 0.75])

def assign_prestige(score):
    if score < q25: return 'Emerging'
    if score < q50: return 'Rising'
    if score < q75: return 'Established'
    return 'A-list'

actor_stats['prestige_tier'] = actor_stats['prestige_score'].apply(assign_prestige)

tier_map = actor_stats['prestige_tier'].to_dict()
df_clean['prestige_tier'] = df_clean['lead_actor'].map(tier_map).fillna('Emerging')

print('Prestige tier distribution:')
print(df_clean['prestige_tier'].value_counts().to_string())
print(f'\nTop 10 actors by avg revenue (adj.):')
top10 = actor_stats.sort_values('avg_rev', ascending=False).head(10)
print(top10[['films', 'avg_rev', 'prestige_tier']].to_string())

# Export lookup for Streamlit app
lookup = actor_stats['prestige_tier'].to_dict()
with open('data/actor_prestige_lookup.json', 'w') as f:
    json.dump(lookup, f, indent=2)
print('\nExported data/actor_prestige_lookup.json')

Prestige tier distribution:
prestige_tier
Emerging       1088
A-list          825
Established     764
Rising          601

Top 10 actors by avg revenue (adj.):
                   films       avg_rev prestige_tier
lead_actor                                          
Idina Menzel           2  1.755596e+09        A-list
Albert Brooks          2  1.479405e+09        A-list
Daisy Ridley           2  1.373711e+09        A-list
Sam Worthington        7  1.371661e+09        A-list
Craig T. Nelson        2  1.305435e+09        A-list
Chiwetel Ejiofor       2  1.149393e+09        A-list
Daniel Radcliffe      11  1.115448e+09        A-list
Robert Downey Jr.     14  1.100712e+09        A-list
Amy Poehler            3  9.924851e+08        A-list
Auliʻi Cravalho        2  9.827047e+08        A-list

Exported data/actor_prestige_lookup.json


In [29]:
# Save featured dataset to FEAT_PATH
cols_to_keep = [
    'tmdb_id', 'title', 'release_date', 'release_year', 'release_window',
    'genres', 'genre_bn', 'budget', 'revenue', 'budget_adj', 'revenue_adj',
    'profit_ratio', 'outcome_label', 'budget_tier', 'prestige_tier',
    'lead_actor', 'cast_names', 'cast_ids', 'runtime', 'vote_average',
    'vote_count', 'popularity',
]
df_feat = df_clean[cols_to_keep].copy()
df_feat.to_csv(FEAT_PATH, index=False)
print(f'Saved {len(df_feat)} rows → {FEAT_PATH}')
print(f'Columns: {list(df_feat.columns)}')

Saved 3278 rows → data\movies_featured_v2.csv
Columns: ['tmdb_id', 'title', 'release_date', 'release_year', 'release_window', 'genres', 'genre_bn', 'budget', 'revenue', 'budget_adj', 'revenue_adj', 'profit_ratio', 'outcome_label', 'budget_tier', 'prestige_tier', 'lead_actor', 'cast_names', 'cast_ids', 'runtime', 'vote_average', 'vote_count', 'popularity']


## Section 4 - Exploratory Analysis

## Section 5 - Bayesian Network

## Section 6 - Probabilistic Inference

## Section 7 - Model Comparison

## Section 8 - Sensitivity Analysis

## Section 9 - Conclusion